# Chicago crimes: klassifisering av kriminalitetstype

Notebooken tester hvor godt en lokal språkmodell klarer å sette riktig lovbruddskategori på en politianmeldelse, bare ut fra fritekstfeltet `Police Report`. Hver kjøring logges i MLflow med parametere, metrikker, prompter og én trace per anmeldelse.

Hele eksemplet kan kjøres herfra, ovenfra og ned: notebooken installerer avhengighetene, starter modellserveren (vLLM) og MLflow, kjører evalueringen og stopper tjenestene til slutt.


## Filer

| Fil / mappe | Innhold |
|---|---|
| `chicago_crime.ipynb` | Denne notebooken |
| `chicago_crimes.csv` | Evalueringsdatasettet: 272 rader i 10 kategorier, med fiktive anmeldelser |
| `prompts/system_prompt.md` | Systemprompten: instruksjoner, beslutningsregler og svarformat |
| `prompts/labels.yaml` | Kategoriene og definisjonene som settes inn i systemprompten |



### 1. Hent avhengigheter og kjør i gang vLLM

1. lag virituelt python-miljø for oppgaven:

    ```python3.12 -m venv .venv```

2. Aktiver området og installer avhengigheter:

    ```source .venv/bin/activate```

    ```pip install --upgrade pip```

    ```pip install -q openai mlflow pandas scikit-learn matplotlib pyyaml ipykernel```

    Mulig du må åpne vscode på nytt hvis ikke .venv dukker opp i "select kernels"

3. Installer vllm (hvis ikke allerede gjort):
    ```
    deactivate
    bash
    curl -fsSL https://raw.githubusercontent.com/vllm-project/vllm-metal/main/install.sh | bash
    source ~/.venv-vllm-metal/bin/activate
    ```

4. Sette miljøvariabler:
    

In [ ]:
import os
import sys
from pathlib import Path

HOST = "127.0.0.1"

# vLLM (model server)
VLLM_BIN = Path("~/.venv-vllm-metal/bin/vllm").expanduser()  # created by the vllm-metal installer in step 3
VLLM_MODEL = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
VLLM_PORT = 8000
VLLM_MAX_MODEL_LEN = 4096

# The notebook calls vLLM's OpenAI-compatible API directly. MODEL must be the name vLLM serves
# (the model passed to `vllm serve`). vLLM ignores the API key unless started with --api-key.
VLLM_API_BASE = os.getenv("VLLM_API_BASE", f"http://{HOST}:{VLLM_PORT}/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "none")
MODEL = os.getenv("VLLM_SERVED_MODEL", VLLM_MODEL)

TEMPERATURE = 0.5
MAX_TOKENS = 4000
ENABLE_THINKING = False  
CONCURRENCY = 4      # parallel requests; lower this if the endpoint rate-limits
NUM_RETRIES = 3
MAX_ROWS = None      # e.g. 10 for a quick trial run; None = all rows

# MLflow. Runs are stored in MLFLOW_DIR/mlflow.db and artifacts in MLFLOW_DIR/mlartifacts.
# Always use the same folder, otherwise earlier runs won't show up.
MLFLOW_PORT = 5000
MLFLOW_DIR = Path(".")
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", f"http://{HOST}:{MLFLOW_PORT}")
EXPERIMENT_NAME = "chicago-crimes-eksempel"
RUN_NAME = None      # None lets MLflow generate a name
PROMPT_VERSION = "v1"
ENABLE_TRACING = True  # log each LLM call as an MLflow trace

# Services
START_SERVICES = True  # False if you run vLLM and MLflow in your own terminals
LOG_DIR = Path("logs")

# Data
DATA_PATH = Path("chicago_crimes.csv")
TEXT_COL = "Police Report"
LABEL_COL = "Primary Type"
ID_COL = "Case Number"

# The notebook must run from the main folder. Check that the project files are found.
missing = [p for p in (DATA_PATH, Path("prompts")) if not p.exists()]
assert not missing, f"Fant ikke {missing} i {Path.cwd()}. Kjør notebooken fra mappen pit-rekrutt."

### 2. Start vLLM og MLFlow

Ved å kjøre cella under, så startes vLLM og MLFlow i egne prosesser.

MLFlow vil være tilgjengelig på adressen: http://127.0.0.1:5000 

In [ ]:
import time
import urllib.error
import urllib.request

LOG_DIR.mkdir(exist_ok=True)
SERVICES = globals().get("SERVICES", {})  # processes started by this notebook, stopped in step 8
BIN_DIR = Path(sys.executable).parent      # mlflow is installed in the notebook kernel's venv

def is_up(url):
    """True if a server answers on url (any response below 500 counts)."""
    try:
        with urllib.request.urlopen(url, timeout=2) as resp:
            return resp.status < 500
    except urllib.error.HTTPError as e:
        return e.code < 500
    except Exception:
        return False

def start_service(name, cmd, health_url, timeout=120):
    """Start cmd in the background unless something already answers on health_url, then wait until it does."""
    if is_up(health_url):
        print(f"{name}: kjører allerede ({health_url})")
        return
    if not START_SERVICES:
        raise RuntimeError(f"{name} svarer ikke på {health_url}. Start den i en terminal, eller sett START_SERVICES = True.")
    log_path = LOG_DIR / f"{name}.log"
    log = open(log_path, "a")
    # start_new_session: the service keeps running if the kernel restarts, and can be stopped as a group.
    proc = subprocess.Popen([str(c) for c in cmd], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    SERVICES[name] = proc
    print(f"{name}: starter (logg: {log_path}) ...", end="", flush=True)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"{name} stoppet med kode {proc.returncode}. Se {log_path}.")
        if is_up(health_url):
            print(" klar")
            return
        time.sleep(2)
    raise TimeoutError(f"{name} svarte ikke innen {timeout} s. Se {log_path}.")

# vLLM can take several minutes the first time, while the model downloads.
start_service("vllm",
              [VLLM_BIN, "serve", VLLM_MODEL, "--max-model-len", VLLM_MAX_MODEL_LEN,
               "--host", HOST, "--port", VLLM_PORT],
              f"http://{HOST}:{VLLM_PORT}/v1/models", timeout=900)

start_service("mlflow",
              [BIN_DIR / "mlflow", "server", "--host", HOST, "--port", MLFLOW_PORT,
               "--backend-store-uri", f"sqlite:///{(MLFLOW_DIR / 'mlflow.db').resolve()}",
               "--artifacts-destination", (MLFLOW_DIR / "mlartifacts").resolve()],
              f"{MLFLOW_TRACKING_URI}/health", timeout=120)

from IPython.display import Markdown, display
display(Markdown(f"**MLflow-grensesnitt:** [{MLFLOW_TRACKING_URI}]({MLFLOW_TRACKING_URI}) (åpnes i en ny fane)"))

### 3. Test om vLLM fungerer
Sender et enkelt prompt til vLLM det den ber om å få svaret "hei"

In [ ]:
import json

# Check that vLLM serves the model and answers a chat request.
with urllib.request.urlopen(f"{VLLM_API_BASE}/models", timeout=10) as resp:
    served = [m["id"] for m in json.load(resp)["data"]]
print("vLLM-modeller:", served)
assert MODEL in served, f"vLLM serverer ikke {MODEL!r}. Sett MODEL til en av {served}."

request = urllib.request.Request(
    f"{VLLM_API_BASE}/chat/completions",
    data=json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": "Svar med ett ord: hei"}],
        "max_tokens": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    }).encode(),
    headers={"Authorization": f"Bearer {VLLM_API_KEY}", "Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=120) as resp:
    print("Svar fra vLLM:", json.load(resp)["choices"][0]["message"]["content"])

### 4. Les inn kategorier og mal for systemprompt

Systemprompten og kategoridefinisjonene ligger i egne filer:

- `prompts/system_prompt.md`: instruksjoner, beslutningsregler og svarformat. `{label_block}` markerer hvor listen over kategorier settes inn.
- `prompts/labels.yaml`: én linje per kategori, på formen `KATEGORI: definisjon`.

Kategoriene i `labels.yaml` bør være de samme som i datasettet. Steg 5 stopper med en feilmelding hvis datasettet inneholder en kategori som mangler i `labels.yaml`.

In [ ]:
import yaml

PROMPTS_DIR = Path("prompts")
SYSTEM_PROMPT_PATH = PROMPTS_DIR / "system_prompt.md"
LABELS_PATH = PROMPTS_DIR / "labels.yaml"

# {label_block} in the system prompt file is replaced with the label definitions.
SYSTEM_PROMPT_TEMPLATE = SYSTEM_PROMPT_PATH.read_text().strip()
LABELS = yaml.safe_load(LABELS_PATH.read_text())

USER_PROMPT_TEMPLATE = """Police report:
<<<
{report}
>>>"""

def render_system_prompt(labels=LABELS):
    label_block = "\n".join(f"- {name}: {desc}" for name, desc in labels.items())
    return SYSTEM_PROMPT_TEMPLATE.replace("{label_block}", label_block)

SYSTEM_PROMPT = render_system_prompt()
print(SYSTEM_PROMPT)

### 5. Last inn data fra "chicago_crimes.csv"

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
if MAX_ROWS:
    df = df.head(MAX_ROWS)

unknown = set(df[LABEL_COL]) - set(LABELS)
assert not unknown, f"Labels missing from LABELS: {unknown}"

print(f"{len(df)} rows")
df[LABEL_COL].value_counts()

## 6. Eksperimentoppsett

Cellen under definerer klassifiseringen og kjører en test på én rad. Sjekk at `predicted` har en gyldig kategori før du kjører hele datasettet i steg 7.

In [ ]:
import json, re
import mlflow
import time
from openai import OpenAI

# OpenAI-compatible client pointed straight at vLLM; retries transient errors itself.
client = OpenAI(base_url=VLLM_API_BASE, api_key=VLLM_API_KEY, max_retries=NUM_RETRIES)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
if ENABLE_TRACING:
    mlflow.openai.autolog()

def parse_label(text):
    """Return (label, rationale); label is None if no valid label is found."""
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        obj = json.loads(cleaned)
        label = str(obj.get("primary_type", "")).strip().upper()
        if label in LABELS:
            return label, obj.get("rationale", "")
    except (json.JSONDecodeError, AttributeError):
        pass
    # Fallback: a "primary_type" field inside otherwise malformed or wrapped JSON.
    # Don't match labels anywhere in the text: reasoning mentions many labels, which gives wrong predictions.
    m = re.findall(r'"primary_type"\s*:\s*"([^"]+)"', text)
    label = m[-1].strip().upper() if m else None
    return (label if label in LABELS else None), ""

def categorize(report, true_label=None, case_number=None):
    start = time.perf_counter()
    result = {"predicted": None, "rationale": "", "raw_response": "", "error": "",
              "prompt_tokens": 0, "completion_tokens": 0}
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT_TEMPLATE.format(report=report)},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
        )
        text = resp.choices[0].message.content or ""
        result["predicted"], result["rationale"] = parse_label(text)
        result["raw_response"] = text
        usage = getattr(resp, "usage", None)
        result["prompt_tokens"] = getattr(usage, "prompt_tokens", 0) or 0
        result["completion_tokens"] = getattr(usage, "completion_tokens", 0) or 0
    except Exception as e:
        result["error"] = repr(e)
    result["latency_s"] = time.perf_counter() - start
    if ENABLE_TRACING:
        _tag_trace(result, true_label, case_number)
    return result

def _tag_trace(result, true_label, case_number):
    """Tag the active trace with true/predicted labels so traces can be filtered in the MLflow UI."""
    span = mlflow.get_current_active_span()
    if span is None:
        return
    result["trace_id"] = span.trace_id
    tags = {"predicted_label": result["predicted"] or "INVALID"}
    if true_label is not None:
        tags["true_label"] = true_label
        tags["correct"] = str(result["predicted"] == true_label).lower()
    if case_number is not None:
        tags["case_number"] = case_number
    if result["error"]:
        tags["error"] = "true"
    mlflow.update_current_trace(tags=tags)

# One root span per row, with the OpenAI call to vLLM (autologged) nested under it.
if ENABLE_TRACING:
    categorize = mlflow.trace(categorize, name="categorize", span_type="CHAIN")

# Smoke test on one row before running the full set.
sample = df.iloc[0]
print("Expected:", sample[LABEL_COL])
categorize(sample[TEXT_COL], sample[LABEL_COL], sample[ID_COL])

## 7. Oppgave

Oppgaven består i å kjøre eksperimenter på "chicago crimes"-datasettet der målet er å finne et oppsett som fungerer best mulig. Det vil si å optimalisere metrikkene som logges per eksperiment.

1. Bli kjent med MLFlow:
    - Kjør eksperiment med det gjeldende oppsettet
    - Gå til MLFlow UI (http://127.0.0.1:5000)
    - Sjekk "Traces" og undersøk noen av enkeltkjøringene
    - Gå til "Model training" og sjekk metrikkene som ble logget i eksperimentet "chicago-crimes-eksempel"


2. Endre modellen fra den lille Qwen2.5-modellen til denne "mlx-community/Qwen3.5-4B-MLX-4bit" (stopp vllm-prosessen først med ```pkill -f "vllm serve"```)
    - Sjekk traces og metrikker tilsvarende over


3. Prøv å se om du kan forbedre metrikkene ved å endre klassedefinisjonene i "labels.yaml". 
    - Bruk chicago_crimes_utv.csv til å gjøre justeringene for å hindre at tilpasser metrikker til testdata
    - Dersom du vil teste klassifisering på utv-data: Lag et nytt eksperiment for å hindre at du blander sammen eksperimenter 


4. Øk temperaturen på modellen f.eks. til 0.5 og kjør et nytt eksperiment
    - Hvordan forklarer du resultatet?

Hvis tid:

5. Kan du finne andre måter å forbedre klassifiseringen? Vær kreativ!

6. Kjør nytt eksperiment, men nå med `"ENABLE_THINKING=True"` 

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import hashlib
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params({
        "model": MODEL,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "enable_thinking": ENABLE_THINKING,
        "prompt_version": PROMPT_VERSION,
        "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:12],
        "n_labels": len(LABELS),
        "n_rows": len(df),
        "data_path": str(DATA_PATH),
    })
    mlflow.log_input(mlflow.data.from_pandas(df, source=str(DATA_PATH), name="chicago_crimes", targets=LABEL_COL), context="evaluation")
    mlflow.log_text(SYSTEM_PROMPT, "prompts/system_prompt.txt")
    mlflow.log_artifact(str(SYSTEM_PROMPT_PATH), "prompts/source")
    mlflow.log_artifact(str(LABELS_PATH), "prompts/source")
    mlflow.log_text(USER_PROMPT_TEMPLATE, "prompts/user_prompt_template.txt")

    with ThreadPoolExecutor(CONCURRENCY) as pool:
        results = list(pool.map(categorize, df[TEXT_COL], df[LABEL_COL], df[ID_COL]))

    preds = pd.concat([df[[ID_COL, LABEL_COL, TEXT_COL]].reset_index(drop=True), pd.DataFrame(results)], axis=1)
    preds["correct"] = preds["predicted"] == preds[LABEL_COL]
    y_true = preds[LABEL_COL]
    y_pred = preds["predicted"].fillna("INVALID")

    # Precision, recall and F1 over the true labels; INVALID predictions count as misses.
    eval_labels = sorted(y_true.unique())
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", labels=eval_labels, zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", labels=eval_labels, zero_division=0)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1,
        "mean_latency_s": preds["latency_s"].mean(),
        "total_prompt_tokens": int(preds["prompt_tokens"].sum()),
        "total_completion_tokens": int(preds["completion_tokens"].sum()),
    }
    mlflow.log_metrics(metrics)

    if ENABLE_TRACING and "trace_id" in preds:
        traced = preds.dropna(subset=["trace_id"])
        # Traces are exported asynchronously; wait until they reach the server before attaching assessments.
        mlflow.flush_trace_async_logging()

        def _with_retry(fn, attempts=10, delay=1.0, **kwargs):
            # The server can take a moment to persist a trace after receiving it, so retry "not found".
            for attempt in range(attempts):
                try:
                    return fn(**kwargs)
                except mlflow.exceptions.MlflowException as e:
                    if "RESOURCE_DOES_NOT_EXIST" not in str(e) or attempt == attempts - 1:
                        raise
                    time.sleep(delay)

        for trace_id, true_label, predicted in zip(traced["trace_id"], traced[LABEL_COL], traced["predicted"]):
            # Ground truth and correctness as assessments, shown on each trace in the MLflow UI.
            _with_retry(mlflow.log_expectation, trace_id=trace_id, name="primary_type", value=true_label)
            _with_retry(mlflow.log_feedback, trace_id=trace_id, name="correct", value=bool(predicted == true_label))
        # Traces are created in worker threads, outside the run's context, so link them to the run explicitly.
        trace_ids = traced["trace_id"].tolist()
        for i in range(0, len(trace_ids), 100):
            mlflow.MlflowClient().link_traces_to_run(trace_ids[i:i + 100], run_id=run.info.run_id)

    report = classification_report(y_true, y_pred, zero_division=0)
    mlflow.log_text(report, "classification_report.txt")
    mlflow.log_table(preds, "predictions.json")

    shown = sorted(set(y_true) | set(y_pred))
    fig, ax = plt.subplots(figsize=(14, 12))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=shown), display_labels=shown).plot(
        ax=ax, xticks_rotation=90, colorbar=False)
    ax.set_title("Primary type: true vs predicted")
    fig.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    print(f"Run {run.info.run_id} logged to {MLFLOW_TRACKING_URI}")

print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()}, indent=2))
print(report)

/Users/kbt007/src/evaluering/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


### Feilklassifiseringer

In [ ]:
errors = preds.loc[~preds["correct"], [ID_COL, LABEL_COL, "predicted", "rationale", TEXT_COL, "error"]]
print(f"{len(errors)} of {len(preds)} rows wrong")
pd.set_option("display.max_colwidth", 200)
errors

8 of 172 rows wrong


,Case Number,Primary Type,predicted,rationale,Police Report,error
10,JK451772,CRIMINAL DAMAGE,BURGLARY,"The victim's vehicle was entered without authority through a broken window, and while no items were stolen, the unlawful entry to commit a theft (even if unsuccessful) constitutes burglary.",The victim reported that at about 5:35 PM they found the rear side window of their vehicle broken while it was parked on the street in the 025XX block of W Fitch Ave. A long scratch ran along the ...,
23,JK192954,OTHER OFFENSE,BURGLARY,"The former husband entered the property without authority to commit a theft or felony inside, which fits the definition of burglary even though no items were taken.","At about 9:00 AM, the victim called officers to their residence in the 057XX block of N Kingsdale Ave and showed them a court order barring their former husband from the home. The victim stated th...",
26,JK194485,CRIMINAL DAMAGE,BURGLARY,"The victim's vehicle was entered without authority, evidenced by the smashed windshield and dented hood, to steal items (stereo and sunglasses), which fits the definition of burglary.","At about 1:17 AM, officers responded to the 117XX block of S Normal Ave after a resident reported hearing glass breaking. The resident's parked vehicle had a smashed windshield and a dented hood. ...",
37,JK125833,MOTOR VEHICLE THEFT,THEFT,"The victim reported their SUV was missing from the street with no broken glass, no tow record, and no suspect identified, indicating the vehicle was likely left unattended rather than forcibly ent...","At about 10:30 PM, the victim reported that their SUV was gone from the spot where they parked it on the street in the 044XX block of N Kenton Ave. The victim's spouse confirmed they had not moved...",
82,JK208595,ROBBERY,BATTERY,"The suspect used physical force (shoving the victim to the ground) to take property, which constitutes battery rather than theft or robbery.","At about 7:30 AM, the victim was walking to a bus stop on the street in the 004XX block of S Damen Ave when a person came up from behind, shoved them to the ground, and pulled a purse from their s...",
107,JK194379,CRIMINAL DAMAGE,BATTERY,"The boyfriend physically threw an object at the victim and ripped a door off a cabinet, causing damage to property and creating a situation of physical contact and harm, which fits the definition ...","At about 10:03 PM, the victim reported that their boyfriend, while moving out of their shared apartment in the 017XX block of W 89th St, threw a laptop against the wall and ripped a door off the k...",
117,JK391498,THEFT,CRIMINAL DAMAGE,"The victim's catalytic converter was cut from their parked pickup truck without the doors being opened or items being taken from inside, which constitutes damage to property without theft of the v...","At about 8:20 PM, the victim reported that a catalytic converter had been cut from their pickup while it was parked in a non-residential parking lot in the 044XX block of S Cottage Grove Ave. The ...",
142,JK383108,BURGLARY,THEFT,"The victim found an unlocked kitchen window left wide open with muddy prints inside, and items were stolen without any forced entry or physical contact, fitting the definition of theft from an unl...","The victim reported that on or about August 16, 2026, they came home to their residence in the 087XX block of S Throop St and found a kitchen window that had been left unlocked wide open and muddy...",
